# FraudIA Claims — 01: Exploración de Datos

Análisis exploratorio del dataset sintético de siniestros.

**Dataset:** 500 siniestros · 174 asegurados · 33 proveedores · 1263 documentos  
**Objetivo:** Entender distribuciones, detectar patrones y validar la calidad de los datos antes del modelado.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

# Ajusta si ejecutas desde Colab
DATA_DIR = Path('../data/processed')

print('✓ Setup OK')

## 1. Carga de datos

In [ ]:
df = pd.read_csv(DATA_DIR / 'claims_with_documents.csv')
scored = pd.read_csv(DATA_DIR / 'claims_scored.csv')
df = df.merge(scored[['id_siniestro','score_riesgo','nivel_riesgo','score_reglas']], on='id_siniestro', how='left')

print(f'Siniestros  : {len(df)}')
print(f'Columnas    : {df.shape[1]}')
print(f'Asegurados  : {df["id_asegurado"].nunique()}')
print(f'Proveedores : {df["id_proveedor"].nunique()}')
df.head(3)

## 2. Distribución por ramo y estado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Ramo
ramo_counts = df['ramo'].value_counts()
axes[0].bar(ramo_counts.index, ramo_counts.values, color=['#1e3a5f','#3b82f6','#93c5fd'])
axes[0].set_title('Siniestros por Ramo', fontsize=13, fontweight='bold')
axes[0].set_ylabel('N° siniestros')
for i, v in enumerate(ramo_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=11)

# Estado
estado_counts = df['estado'].value_counts().head(6)
axes[1].barh(estado_counts.index, estado_counts.values, color='#3b82f6')
axes[1].set_title('Siniestros por Estado', fontsize=13, fontweight='bold')
axes[1].set_xlabel('N° siniestros')

plt.tight_layout()
plt.savefig('../data/outputs/01_distribucion_ramo_estado.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Distribución de montos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['monto_reclamado'].dropna(), bins=40, color='#1e3a5f', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución Monto Reclamado', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Monto ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

df_ratio = df['ratio_monto_suma'].dropna()
axes[1].hist(df_ratio, bins=40, color='#dc2626', edgecolor='white', alpha=0.85)
axes[1].set_title('Ratio Monto Reclamado / Suma Asegurada', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ratio (0-1)')
axes[1].axvline(0.95, color='orange', linestyle='--', label='Umbral alerta (0.95)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/outputs/01_distribucion_montos.png', dpi=120, bbox_inches='tight')
plt.show()

print(df[['monto_reclamado','monto_estimado','monto_pagado']].describe().round(0))

## 4. Distribución del score de riesgo y semáforo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma del score
color_map = {'BAJO': '#16a34a', 'MEDIO': '#d97706', 'ALTO': '#dc2626'}
for nivel, grp in df.groupby('nivel_riesgo'):
    axes[0].hist(grp['score_riesgo'], bins=20, alpha=0.7,
                 label=nivel, color=color_map.get(nivel, 'gray'))
axes[0].set_title('Distribución Score de Riesgo por Nivel', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Score (0-100)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(40, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(75, color='gray', linestyle='--', alpha=0.5)
axes[0].legend()

# Pie del semáforo
nivel_counts = df['nivel_riesgo'].value_counts().reindex(['BAJO','MEDIO','ALTO'])
axes[1].pie(nivel_counts.values, labels=[f'{n}\n({v} casos)' for n,v in nivel_counts.items()],
            colors=['#16a34a','#d97706','#dc2626'], autopct='%1.1f%%',
            startangle=90, pctdistance=0.75)
axes[1].set_title('Proporción por Nivel de Riesgo', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/outputs/01_distribucion_score.png', dpi=120, bbox_inches='tight')
plt.show()

print(nivel_counts)
print(f"\nScore medio: {df['score_riesgo'].mean():.1f}  |  Máximo: {df['score_riesgo'].max():.1f}")

## 5. Señales de fraude — frecuencia de alertas

In [ ]:
alert_cols = [
    'alerta_borde_inicio','alerta_borde_fin','reporte_tardio',
    'narrativa_similar','narrativa_clonada','proveedor_lista_restrictiva',
    'doc_factura_alterada','doc_ruc_invalido','doc_sin_denuncia_previa',
    'doc_parte_tardio','doc_sin_testigos','doc_robo','doc_perdida_total',
]
alert_cols = [c for c in alert_cols if c in df.columns]

alert_freq = df[alert_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#dc2626' if v > 50 else '#d97706' if v > 20 else '#1e3a5f' for v in alert_freq.values]
plt.barh(alert_freq.index, alert_freq.values, color=colors)
plt.title('Frecuencia de Señales de Riesgo en el Dataset', fontsize=13, fontweight='bold')
plt.xlabel('N° siniestros con la señal activa')
for i, v in enumerate(alert_freq.values):
    plt.text(v + 1, i, str(int(v)), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../data/outputs/01_frecuencia_alertas.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Top 10 siniestros de mayor riesgo

In [ ]:
top10 = df.nlargest(10, 'score_riesgo')[[
    'id_siniestro','ramo','nivel_riesgo','score_riesgo',
    'monto_reclamado','id_proveedor','id_asegurado'
]].reset_index(drop=True)
top10.index += 1
print('Top 10 siniestros por score de riesgo:')
top10

## 7. Análisis de proveedores

In [ ]:
prov_stats = df.groupby('id_proveedor').agg(
    n_siniestros=('id_siniestro','count'),
    score_medio=('score_riesgo','mean'),
    n_alto=('nivel_riesgo', lambda x: (x == 'ALTO').sum()),
    monto_total=('monto_reclamado','sum')
).sort_values('n_alto', ascending=False).head(10)

print('Top 10 proveedores con más siniestros ALTO riesgo:')
prov_stats.round(1)

## 8. Correlación entre variables numéricas

In [ ]:
num_cols = [
    'score_riesgo','monto_reclamado','ratio_monto_suma','similitud_narrativa',
    'dias_desde_inicio_poliza','dias_ocurrencia_reporte','historial_siniestros_asegurado',
    'n_reclamos_12_meses','n_reclamos_historico'
]
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols, fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title('Matriz de Correlación — Variables de Riesgo', fontsize=13, fontweight='bold')
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        val = corr.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if abs(val) > 0.5 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig('../data/outputs/01_correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Resumen estadístico

> Los datos sintéticos reproducen patrones realistas del sector asegurador. El 1% de siniestros en nivel ALTO concentra las señales más críticas. Ver `02_modelo_fraude.ipynb` para el entrenamiento.

In [ ]:
print('=== RESUMEN DEL DATASET ===')
print(f'Total siniestros      : {len(df)}')
print(f'Score promedio        : {df["score_riesgo"].mean():.1f}')
print(f'Nivel BAJO  (0-40)    : {(df["nivel_riesgo"]=="BAJO").sum()} ({(df["nivel_riesgo"]=="BAJO").mean():.1%})')
print(f'Nivel MEDIO (41-75)   : {(df["nivel_riesgo"]=="MEDIO").sum()} ({(df["nivel_riesgo"]=="MEDIO").mean():.1%})')
print(f'Nivel ALTO  (76-100)  : {(df["nivel_riesgo"]=="ALTO").sum()} ({(df["nivel_riesgo"]=="ALTO").mean():.1%})')
print(f'Con proveedor lista restrictiva : {df["proveedor_lista_restrictiva"].sum() if "proveedor_lista_restrictiva" in df.columns else "N/A"}')
print(f'Con narrativa clonada           : {df["narrativa_clonada"].sum() if "narrativa_clonada" in df.columns else "N/A"}')
print(f'Con factura alterada            : {df["doc_factura_alterada"].sum() if "doc_factura_alterada" in df.columns else "N/A"}')